In [5]:
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# -------------------------------------------------
# Session / config
# -------------------------------------------------
session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()

bucket = "propelnoi-vijay-datalake"

train_s3 = f"s3://{bucket}/modeling/noi_deepar/train/"
test_s3 = f"s3://{bucket}/modeling/noi_deepar/test/"
inference_s3 = f"s3://{bucket}/modeling/noi_deepar/inference_input/"
training_output_s3 = f"s3://{bucket}/model-artifacts/noi_deepar/"
forecast_output_s3 = f"s3://{bucket}/modeling/noi_deepar/inference_output/"

print("Region:", region)
print("Role:", role)
print("Train path:", train_s3)
print("Test path:", test_s3)
print("Inference path:", inference_s3)
print("Training artifacts path:", training_output_s3)
print("Forecast output path:", forecast_output_s3)

# -------------------------------------------------
# DeepAR container
# -------------------------------------------------
container = image_uris.retrieve(
    framework="forecasting-deepar",
    region=region
)

print("DeepAR container:", container)

# -------------------------------------------------
# Estimator
# -------------------------------------------------
estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=training_output_s3,
    sagemaker_session=session
)

# -------------------------------------------------
# Hyperparameters
# -------------------------------------------------
estimator.set_hyperparameters(
    time_freq="M",
    prediction_length="12",
    context_length="12",
    epochs="30",
    early_stopping_patience="10",
    mini_batch_size="32",
    learning_rate="0.001",
    num_layers="2",
    num_cells="40",
    likelihood="gaussian"
)

# -------------------------------------------------
# Training inputs
# -------------------------------------------------
train_input = TrainingInput(
    s3_data=train_s3,
    content_type="json"
)

test_input = TrainingInput(
    s3_data=test_s3,
    content_type="json"
)

# -------------------------------------------------
# Train DeepAR model
# -------------------------------------------------
estimator.fit({
    "train": train_input,
    "test": test_input
})

print("Training completed.")
print("Model artifact:", estimator.model_data)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: forecasting-deepar-2026-03-08-01-41-18-699


2026-03-08 01:41:19 Starting - Starting the training job...
2026-03-08 01:41:32 Starting - Preparing the instances for training...
2026-03-08 01:41:56 Downloading - Downloading input data...
2026-03-08 01:42:36 Downloading - Downloading the training image...........Docker entrypoint called with argument(s): train
Running default environment configuration script
Running custom environment configuration script
/opt/amazon/lib/python3.9/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
[03/08/2026 01:44:33 INFO 139814752233280] Reading default configuration from /opt/amazon/lib/python3.9/site-packages/algorithm/resources/default-input.json: {'_kvstore': 'auto', '_num_gpus': 'auto', '_num_kv_servers': 'auto', '_tuning_objective_metric': '', 'cardinality': 'auto', 'dropout_rate': '0.10', 'early_stopping_patience': '', 'embedding_dimension': '10', 'learning_rate': '0.001', 'likelihood': 'student-t', 'mini_b

In [24]:
transformer = estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/modeling/noi_deepar/inference_output/",
    accept="application/jsonlines",
    strategy="SingleRecord"
)

transformer.transform(
    data=f"s3://{bucket}/modeling/noi_deepar/inference_input/",
    content_type="application/jsonlines",
    split_type="Line"
)

transformer.wait()

INFO:sagemaker:Creating model with name: forecasting-deepar-2026-03-08-03-51-31-417
INFO:sagemaker:Creating transform job with name: forecasting-deepar-2026-03-08-03-51-31-933


......................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
Running custom environment configuration script
/opt/amazon/lib/python3.9/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
Failed to set debug level to 20, using INFO
[03/08/2026 03:57:57 INFO 139671338542912] Estimated memory required per model 7.797956466674805MB.
[03/08/2026 03:57:57 INFO 139671338542912] Estimated available memory 6821.84549331665MB.
[03/08/2026 03:57:57 INFO 139671338542912] Estimated maximum number of workers for the available memory is 874.
[03/08/2026 03:57:57 INFO 139671338542912] Using 2 workers
[03/08/2026 03:57:57 INFO 139671338542912] loading entry points
[03/08/2026 03:57:57 INFO 139671338542912] Prediction endpoint operating in batch mode
[03/08/2026 03:57:57 INFO 139671338542912] loaded request iterator application/jsonlines
[03/08/20